In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio

# run in a cell (reinstall torch + torchvision + torchaudio from cu128 index)
!pip install --upgrade --force-reinstall --no-deps \
  --index-url https://download.pytorch.org/whl/cu128 \
  torch torchvision torchaudio

In [ ]:
    # import os
    # file_to_delete = "/kaggle/working/whisper_comparison.csv"
    # if os.path.exists(file_to_delete):
    #     os.remove(file_to_delete)
    #     print(f"Deleted: {file_to_delete}")
    # else:
    #     print(f"File not found: {file_to_delete}")

In [2]:
# ============================================================
# 🔍 Whisper Large-v3 vs Fine-tuned Whisper Large-v3 (WER Comparison)
# Incremental Saving + Per-sample WER (FIXED)
# ============================================================

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
from jiwer import wer
import pandas as pd
import torchaudio
from tqdm import tqdm
import os

# ============================================================
# ⚙️ 1. Configuration
# ============================================================
dataset_name = "MINERVA-TEAM/minerva-ar-en-edu-codeswitch-dataset"       
model_name_finetuned = "tokakhaled24/whisper-minerva-medium-fine-tuned"  
split = "test"
save_path = "/kaggle/working/whisper_comparison_medium.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 📦 2. Load Dataset (with proper audio handling)
# ============================================================
dataset = load_dataset(dataset_name, split=split)
# Cast to Audio with sampling_rate=16000 for Whisper
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))  
print(f"Loaded dataset: {len(dataset)} samples")

# ============================================================
# 🧠 3. Load Processor & Models
# ============================================================
processor = WhisperProcessor.from_pretrained("openai/whisper-medium")

model_base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to(device)
model_finetuned = WhisperForConditionalGeneration.from_pretrained(model_name_finetuned).to(device)

# ============================================================
# 🎧 4. Transcription Helper (using decoded audio array)
# ============================================================
def transcribe(model, audio_dict):
    """
    Transcribe audio using already-decoded audio array from dataset.
    audio_dict contains 'array' and 'sampling_rate' keys.
    """
    waveform = audio_dict["array"]
    sr = audio_dict["sampling_rate"]
    
    # Ensure mono and correct sample rate
    if len(waveform.shape) > 1:
        waveform = waveform.mean(axis=0)
    
    # Process audio
    input_features = processor(
        waveform, 
        sampling_rate=sr, 
        return_tensors="pt"
    ).input_features.to(device)
    
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
    
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].lower().strip()

# ============================================================
# 💾 5. Resume from previous progress (if exists)
# ============================================================
if os.path.exists(save_path):
    existing_df = pd.read_csv(save_path)
    processed_count = len(existing_df)
    print(f"Resuming from saved progress — {processed_count} samples already processed.")
else:
    existing_df = pd.DataFrame(columns=[
        "reference", 
        "base_prediction", 
        "finetuned_prediction", 
        "wer_base", 
        "wer_finetuned"
    ])
    processed_count = 0
    print("No previous progress found — starting fresh.")

# ============================================================
# 🧪 6. Loop through dataset with incremental saving
# ============================================================
batch_size = 10  # save every 10 rows
buffer = []

for i, sample in enumerate(tqdm(dataset, desc="Evaluating", initial=processed_count), start=processed_count):
    if i < processed_count:
        continue  # skip already processed samples

    ref_text = sample["text"].lower().strip()
    audio_data = sample["audio"]  # This is a dict with 'array' and 'sampling_rate'

    pred_base = transcribe(model_base, audio_data)
    pred_ft = transcribe(model_finetuned, audio_data)

    # compute per-sample WERs
    wer_base_sample = wer([ref_text], [pred_base])
    wer_ft_sample = wer([ref_text], [pred_ft])

    buffer.append({
        "reference": ref_text,
        "base_prediction": pred_base,
        "finetuned_prediction": pred_ft,
        "wer_base": wer_base_sample,
        "wer_finetuned": wer_ft_sample
    })

    # Save every 10 samples (or last few)
    if len(buffer) >= batch_size or i == len(dataset) - 1:
        temp_df = pd.DataFrame(buffer)
        existing_df = pd.concat([existing_df, temp_df], ignore_index=True)
        existing_df.to_csv(save_path, index=False, encoding="utf-8-sig")
        buffer = []
        print(f"✅ Saved progress up to sample {i+1}/{len(dataset)}")

# ============================================================
# 📊 7. Compute Overall WER at the End
# ============================================================
refs = existing_df["reference"].tolist()
preds_base = existing_df["base_prediction"].tolist()
preds_ft = existing_df["finetuned_prediction"].tolist()

wer_base_total = wer(refs, preds_base)
wer_ft_total = wer(refs, preds_ft)
improvement = (wer_base_total - wer_ft_total) * 100

print("\n=================== FINAL RESULTS ===================")
print(f"Overall WER (Base Whisper medium): {wer_base_total:.3f}")
print(f"Overall WER (Fine-tuned medium): {wer_ft_total:.3f}")
print(f"Improvement: {improvement:.2f}%")
print("=====================================================")
print(f"Results (with per-sample WER) saved to: {save_path}")

2025-10-24 20:36:51.495787: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761338211.858607     123 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761338211.970455     123 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/313M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/351M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/330M [00:00<?, ?B/s]

...

✅ Saved progress up to sample 370/390


Evaluating:  97%|█████████▋| 380/390 [1:04:22<01:56, 11.66s/it]

✅ Saved progress up to sample 380/390


Evaluating: 100%|██████████| 390/390 [1:06:07<00:00, 10.17s/it]

✅ Saved progress up to sample 390/390

=================== FINAL RESULTS ===================
Overall WER (Base Whisper medium): 1.627
Overall WER (Fine-tuned medium): 0.462
Improvement: 116.59%
Results (with per-sample WER) saved to: /kaggle/working/whisper_comparison_medium.csv
